# 03 - Report figures

Figures for the report, drawn from the saved scores in `experiments/*/results/`. Nothing is trained here: run all cells after new runs finish and the figures update. Each figure is also saved as a PNG in `report_figures/` for the report.

Every variant was trained twice (seeds 0 and 1). The figures show the mean and, as faint dots, each seed, so the reader sees how much of a difference is real and how much is chance.

In [ ]:
import json
import os

import matplotlib.pyplot as plt
import numpy as np

from lulc import CLASSES

BLUE, ORANGE, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e", "#e6e5e1"   # validation, test, text, axes
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": MUTED, "axes.labelcolor": INK,
                     "xtick.color": MUTED, "ytick.color": MUTED, "axes.grid": True, "axes.grid.axis": "y",
                     "grid.color": GRID, "grid.linewidth": 0.8, "axes.axisbelow": True, "font.size": 10})
os.makedirs("report_figures", exist_ok=True)
SPLITS = (("val", BLUE, "validation (10 sites)"), ("test", ORANGE, "test (3 sites)"))


def runs(exp, name):
    """Both seeds of a run, as loaded results files: [seed 0, seed 1]."""
    return [json.load(open(f"experiments/{exp}/results/results_{name}{s}.json")) for s in ("", "_seed1")]


def score(r, split, cls=None):
    """One run's score: split 'val' or 'test'; cls None = mIoU, else that class's IoU."""
    if cls is None:
        return r["val_miou"] if split == "val" else r["miou"]
    return r["val_iou" if split == "val" else "iou"][CLASSES.index(cls)]

## Figure 1: what each step added

The path from the first model on the full dataset to the current best: the baseline (`hybrid`, exp1), plus the yearly median radar as extra input (context, exp2), plus cleaner labels (AGREE: a filled label is kept only where ESA WorldCover agrees, exp3). Left: mean IoU over all classes. Right: water, the class that broke the first prototype.

**Read with care:** context lifts the **test** sites (mIoU +0.03, water 0.42 -> 0.68) but leaves the **validation** sites unchanged (mIoU 0.379 -> 0.380, water 0.57 -> 0.56). Only cleaner labels (AGREE) improve both. Context was chosen as the base after looking at test scores, so for context the test sites are no longer a fully untouched final exam; an independent confirmation needs test sites that were never looked at.

In [ ]:
steps = [("Baseline", runs("exp1_data", "hybrid")),
         ("+ context", runs("exp2_architecture", "hybrid_ctx")),
         ("+ AGREE", runs("exp3_agree_rare", "hybrid_ctx_agree"))]
x = np.arange(len(steps))
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, cls, title in zip(axes, (None, "water"), ("mIoU, all 7 classes", "Water IoU")):
    for split, color, label in SPLITS:
        seeds = np.array([[score(r, split, cls) for r in rs] for _, rs in steps])     # (steps, seeds)
        mean = seeds.mean(1)
        ax.scatter(np.repeat(x, seeds.shape[1]), seeds.ravel(), color=color, s=16, alpha=0.4, zorder=3)
        ax.plot(x, mean, color=color, lw=2, marker="o", ms=7, label=label, zorder=4)
        ax.annotate(f"{mean[-1]:.3f}", (x[-1], mean[-1]), xytext=(9, 0), textcoords="offset points", va="center", color=INK)
    ax.set_xticks(x, [s for s, _ in steps])
    ax.set_xlim(-0.3, len(steps) - 0.45)
    ax.set_title(title, loc="left", color=INK)
axes[0].set_ylabel("IoU (higher is better)")
axes[0].legend(frameon=False, loc="upper left")
fig.suptitle("Context, then cleaner labels: test mIoU 0.335 → 0.404, test water 0.42 → 0.67",
             x=0.01, ha="left", fontsize=12, color=INK)
fig.text(0.01, -0.03, "Lines: mean of 2 training seeds. Faint dots: each seed. Validation = 10 sites used to choose models; "
         "test = 3 sites never used for any choice.", color=MUTED, fontsize=8)
plt.tight_layout()
fig.savefig("report_figures/fig1_progress.png", dpi=200, bbox_inches="tight")
plt.show()

print(f"{'step':12s}{'val mIoU':>10s}{'test mIoU':>11s}{'val water':>11s}{'test water':>12s}")
for name, rs in steps:
    print(f"{name:12s}" + "".join(f"{np.mean([score(r, s, c) for r in rs]):{w}.3f}"
                                  for s, c, w in (("val", None, 10), ("test", None, 11), ("val", "water", 11), ("test", "water", 12))))

## Figure 2: flooded vegetation needs better labels, not more examples

Oversampling drew training windows with flooded vegetation or bare 3x as often, so the model saw about twice as many flooded-vegetation pixels per epoch (2.2% -> 4.6%). AGREE instead removed filled labels that ESA WorldCover contradicts. If the class were short of examples, oversampling should help; if its labels were the problem, AGREE should. All four runs use context.

In [ ]:
cells = [("Context only", runs("exp2_architecture", "hybrid_ctx")),
         ("+ oversampling\n(2x the examples)", runs("exp3_agree_rare", "hybrid_ctx_over3")),
         ("+ AGREE\n(cleaner labels)", runs("exp3_agree_rare", "hybrid_ctx_agree")),
         ("+ both", runs("exp3_agree_rare", "hybrid_ctx_agree_over3"))]
x, w = np.arange(len(cells)), 0.36
fig, ax = plt.subplots(figsize=(9, 4.4))
for i, (split, color, label) in enumerate(SPLITS):
    seeds = np.array([[score(r, split, "flooded_vegetation") for r in rs] for _, rs in cells])
    pos = x + (i - 0.5) * w
    ax.bar(pos, seeds.mean(1), w, color=color, edgecolor="white", linewidth=1.5, label=label)
    ax.scatter(np.repeat(pos, seeds.shape[1]), seeds.ravel(), color=INK, s=12, alpha=0.55, zorder=3)
    if split == "val":                                             # label the decision set only
        for p, m, top in zip(pos, seeds.mean(1), seeds.max(1)):
            ax.annotate(f"{m:.2f}", (p, max(m, top)), xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", color=INK, fontsize=9)
ax.set_xticks(x, [c for c, _ in cells])
ax.set_ylabel("flooded vegetation IoU (higher is better)")
ax.set_ylim(0, None)
ax.legend(frameon=False, loc="upper left")
ax.set_title("Flooded vegetation responds to cleaner labels, not to more examples", loc="left", color=INK, fontsize=12)
fig.text(0.01, -0.04, "Bars: mean of 2 training seeds. Dots: each seed. Numbers: validation means.", color=MUTED, fontsize=8)
plt.tight_layout()
fig.savefig("report_figures/fig2_flooded_vegetation.png", dpi=200, bbox_inches="tight")
plt.show()

for name, rs in cells:
    print(f"{name.splitlines()[0]:16s} val {np.mean([score(r, 'val', 'flooded_vegetation') for r in rs]):.3f}   "
          f"test {np.mean([score(r, 'test', 'flooded_vegetation') for r in rs]):.3f}")